In [1]:
!pip install azure-ai-formrecognizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.4/301.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.0 MB/s eta 0:00:00


In [ ]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential
import json

#
endpoint = "https://mrinaldocintelligence.cognitiveservices.azure.com/"
key = ""

# 📄 File path (upload in Colab first)
file_path = "/content/sample_student_document.pdf"

# 🔌 Create client
client = DocumentAnalysisClient(endpoint, AzureKeyCredential(key))

# 📊 Analyze document using prebuilt model
with open(file_path, "rb") as f:
    poller = client.begin_analyze_document("prebuilt-document", document=f)
    result = poller.result()

# 🧠 Step 1: Extract FULL TEXT (correct way)
full_text = ""

for page in result.pages:
    for line in page.lines:
        full_text += line.content + "\n"

print("📄 Full Extracted Text:\n")
print(full_text)

# 🎯 Step 2: Convert text → structured data (smart parsing)
structured_data = {}

for line in full_text.split("\n"):
    if "Student Name" in line:
        structured_data["student_name"] = line.split(":")[-1].strip()
    elif "Student ID" in line:
        structured_data["student_id"] = line.split(":")[-1].strip()
    elif "Course" in line:
        structured_data["course"] = line.split(":")[-1].strip()
    elif "Date of Issue" in line:
        structured_data["date_of_issue"] = line.split(":")[-1].strip()
    elif "Fee Amount" in line:
        structured_data["fee_amount"] = line.split(":")[-1].strip()

# 📦 Step 3: Save JSON output
with open("output.json", "w") as f:
    json.dump(structured_data, f, indent=4)

# 🖨️ Final Output
print("\n✅ Structured Output:\n")
print(structured_data)

📄 Full Extracted Text:

Student Document
Student Name: Mrinal Mayank
Student ID: 6400955
Course: B.Tech Computer Science
Date of Issue: 15 April 2026
Fee Amount: 50000 INR


✅ Structured Output:

{'student_name': 'Mrinal Mayank', 'student_id': '6400955', 'course': 'B.Tech Computer Science', 'date_of_issue': '15 April 2026', 'fee_amount': '50000 INR'}


In [ ]:
# 🔹 Install required libraries
!pip install azure-ai-formrecognizer azure-data-tables

# 🔹 Imports
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential
from azure.data.tables import TableServiceClient
import os
import json


# 🔐 STEP 1: Add your credentials (IMPORTANT)
endpoint = "https://mrinaldocintelligence.cognitiveservices.azure.com/"
key =  ""
connection_string = "DefaultEndpointsProtocol=https;AccountName=mrinalstudentdocs;AccountKey=2VLxF2IufNz7dpujpwEV9R65fUJ/SSqjS6ljlb+Pv79j9wIUzoxq6Lq36WkGb9oXshcD0OYhZd0g+ASt/nKmjw==;EndpointSuffix=core.windows.net"

# 📄 STEP 2: Auto-detect uploaded file in Colab
files = os.listdir('/content')
pdf_files = [f for f in files if f.endswith('.pdf')]

if not pdf_files:
    raise Exception("❌ No PDF found in /content. Upload file first!")

file_path = "/content/" + pdf_files[0]
print("📄 Using file:", file_path)

# 🔌 STEP 3: Create Document Intelligence client
client = DocumentAnalysisClient(endpoint, AzureKeyCredential(key))

# 📊 STEP 4: Analyze document
with open(file_path, "rb") as f:
    poller = client.begin_analyze_document("prebuilt-document", document=f)
    result = poller.result()

# 🧠 STEP 5: Extract full text
full_text = ""
for page in result.pages:
    for line in page.lines:
        full_text += line.content + "\n"

print("\n📄 Extracted Text:\n", full_text)

# 🎯 STEP 6: Convert to structured JSON
structured_data = {}

for line in full_text.split("\n"):
    if "Student Name" in line:
        structured_data["student_name"] = line.split(":")[-1].strip()
    elif "Student ID" in line:
        structured_data["student_id"] = line.split(":")[-1].strip()
    elif "Course" in line:
        structured_data["course"] = line.split(":")[-1].strip()
    elif "Date of Issue" in line:
        structured_data["date_of_issue"] = line.split(":")[-1].strip()
    elif "Fee Amount" in line:
        structured_data["fee_amount"] = line.split(":")[-1].strip()

print("\n✅ Structured Data:\n", structured_data)

# 📦 STEP 7: Save JSON locally
with open("output.json", "w") as f:
    json.dump(structured_data, f, indent=4)

print("\n💾 JSON saved locally")

# ☁️ STEP 8: Store in Azure Table Storage
table_name = "studentdata"

service = TableServiceClient.from_connection_string(conn_str=connection_string)
table_client = service.create_table_if_not_exists(table_name)

entity = {
    "PartitionKey": "student",
    "RowKey": structured_data.get("student_id", "unknown"),
    "Name": structured_data.get("student_name"),
    "Course": structured_data.get("course"),
    "Date": structured_data.get("date_of_issue"),
    "Fee": structured_data.get("fee_amount")
}

table_client.upsert_entity(entity)

print("\n🚀 Data stored successfully in Azure Table Storage!")

📄 Using file: /content/sample_student_document (1).pdf

📄 Extracted Text:
 Student Document
Student Name: Mrinal Mayank
Student ID: 6400955
Course: B.Tech Computer Science
Date of Issue: 15 April 2026
Fee Amount: 50000 INR


✅ Structured Data:
 {'student_name': 'Mrinal Mayank', 'student_id': '6400955', 'course': 'B.Tech Computer Science', 'date_of_issue': '15 April 2026', 'fee_amount': '50000 INR'}

💾 JSON saved locally

🚀 Data stored successfully in Azure Table Storage!
